### 03.03. Setting up Indexes

"""
Google Gemini Integration Example

This file demonstrates how to use Google Gemini LLM and embeddings.
Original Azure example is in "code_03_XX A basic Agent - Router application.ipynb".
"""

In [1]:
# Install prerequisite packages (Gemini only – Azure removed)
!pip install --upgrade \
    llama-index \
    nest_asyncio \
    python-dotenv \
    llama-index-embeddings-google-genai \
    llama-index-llms-google-genai \
    google-genai

In [ ]:
# Setup Gemini connection (NO Azure, clean, working)

import os
import nest_asyncio

from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

# Fix async issues (VS Code / Jupyter)
nest_asyncio.apply()

# IMPORTANT: use GOOGLE_API_KEY (not GEMINI_API_KEY)
os.environ["GOOGLE_API_KEY"] = ""

# Configure Gemini LLM (supported model)
Settings.llm = GoogleGenAI(model="gemini-2.5-flash")

# Configure Gemini Embeddings (supported model)
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

print("✅ Gemini LLM and Embeddings configured successfully")

✅ Gemini LLM and Embeddings configured successfully


In [ ]:
# Gemini + LlamaIndex setup (modern, supported, VS Code friendly)

import os
import nest_asyncio

nest_asyncio.apply()

# ✅ Use GOOGLE_API_KEY (do NOT use GEMINI_API_KEY anywhere)
os.environ["GOOGLE_API_KEY"] = ""

from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

# Configure LLM and Embeddings (SUPPORTED models)
Settings.llm = GoogleGenAI(model="gemini-2.5-flash")

# Configure Gemini Embeddings (supported model)
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

# Sentence splitter
splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=200)

# ────────────────────────────────────────
# AeroFlow PDF
aeroflow_docs = SimpleDirectoryReader(
    input_files=["AeroFlow_Specification_Document.pdf"]
).load_data()

aeroflow_index = VectorStoreIndex.from_documents(aeroflow_docs, show_progress=True)

aeroflow_query_engine = aeroflow_index.as_query_engine()

# ────────────────────────────────────────
# EcoSprint PDF
ecosprint_docs = SimpleDirectoryReader(
    input_files=["EcoSprint_Specification_Document.pdf"]
).load_data()

ecosprint_index = VectorStoreIndex.from_documents(ecosprint_docs, show_progress=True)

ecosprint_query_engine = ecosprint_index.as_query_engine()

print("✅ Indices created successfully and ready for queries")

/Users/apple/projects/agentic-ai-for-developers-concepts-and-applications-for-enterprises-3913172/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 2/2 [00:00<00:00,  3.97it/s]

✅ Indices created successfully and ready for queries


### 03.04. Setup the Agentic Router

In [4]:
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

# Tool for AeroFlow document
aeroflow_tool = QueryEngineTool.from_defaults(
    query_engine=aeroflow_query_engine,
    name="aeroflow_specs",
    description=(
        "Information about AeroFlow including design, "
        "technical specifications, features, maintenance, and warranty."
    ),
)

# Tool for EcoSprint document
ecosprint_tool = QueryEngineTool.from_defaults(
    query_engine=ecosprint_query_engine,
    name="ecosprint_specs",
    description=(
        "Information about EcoSprint including design, "
        "technical specifications, features, maintenance, and warranty."
    ),
)

# Router that selects the best tool using the configured LLM (GoogleGenAI)
router_agent = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        aeroflow_tool,
        ecosprint_tool,
    ],
    verbose=True,  # optional but VERY helpful for debugging
)

### 03.05. Route with Agentic AI

In [5]:
#Ask a question about NoSQL
response = router_agent.query("What colors are available for AeroFlow?")
print("\nResponse: ",str(response))

2026-02-13 10:40:34,483 - INFO - AFC is enabled with max remote calls: 10.
2026-02-13 10:40:36,656 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-02-13 10:40:36,663 - INFO - Selecting query engine 0: Choice (1) is 'Information about AeroFlow including design, technical specifications, features, maintenance, and warranty.' The question asks about 'AeroFlow', and this choice directly refers to 'AeroFlow'. Information about available colors would typically fall under 'design' or 'features' for a product like AeroFlow..


Selecting query engine 0: Choice (1) is 'Information about AeroFlow including design, technical specifications, features, maintenance, and warranty.' The question asks about 'AeroFlow', and this choice directly refers to 'AeroFlow'. Information about available colors would typically fall under 'design' or 'features' for a product like AeroFlow..


2026-02-13 10:40:36,985 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2026-02-13 10:40:37,019 - INFO - AFC is enabled with max remote calls: 10.
2026-02-13 10:40:37,989 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"



Response:  The AeroFlow is available in Coastal Blue, Sunset Orange, and Pearl White.


In [6]:
response = router_agent.query("What colors are available for EcoSprint?")
print("\nResponse: ",str(response))

2026-02-13 10:40:38,011 - INFO - AFC is enabled with max remote calls: 10.
2026-02-13 10:40:39,935 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-02-13 10:40:39,939 - INFO - Selecting query engine 1: The question asks about 'EcoSprint', and choice (2) explicitly states it provides 'Information about EcoSprint'. Within the scope of 'design' or 'features' mentioned in choice (2), information about available colors would typically be found. Choice (1) is about 'AeroFlow', which is not relevant to the question about 'EcoSprint'..


Selecting query engine 1: The question asks about 'EcoSprint', and choice (2) explicitly states it provides 'Information about EcoSprint'. Within the scope of 'design' or 'features' mentioned in choice (2), information about available colors would typically be found. Choice (1) is about 'AeroFlow', which is not relevant to the question about 'EcoSprint'..


2026-02-13 10:40:40,276 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2026-02-13 10:40:40,308 - INFO - AFC is enabled with max remote calls: 10.
2026-02-13 10:40:41,062 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"



Response:  The EcoSprint is available in Midnight Black, Ocean Blue, and Pearl White.
